# 06b-d — Scaling dell'ottimizzazione e accoppiamento annidato

Un solo esperimento train-only risponde a più domande appaiate: budget, schedule, obiettivo voltage-only contro voltage→STATE, controllo causale rimescolato, gradient probing e trasferimento ricorsivo a 8 ms. I cinque bracci condividono inizializzazione e minibatch; i checkpoint crescenti evitano training duplicati.

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})

## 1. Sorgenti immutabili e dataset

Servono gli artefatti 05t, 06b, 06b-b e 06b-c, il dataset targeted base e il top-up BAP v3. Ogni autorità sperimentale viene identificata tramite SHA-256.

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
from src.hayflow_model.causal_voltage_state_coupling_forensic import EXPECTED_06B_INDEX_SHA256
from src.hayflow_model.causal_voltage_bridge_representation_forensic import EXPECTED_06BB_INDEX_SHA256
from src.hayflow_model.nested_coupling_optimization_scaling_forensic import EXPECTED_06BC_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
def indexed(name,expected,env):
 override=os.environ.get(env);source=discover_indexed_artifact_source(INPUT_ROOT,expected,override=Path(override) if override else None);assert source is not None,f'Artefatto {name} esatto non trovato.';return source
ARTIFACT_05T_SOURCE=indexed('05t',EXPECTED_05T_INDEX_SHA256,'HAYFLOW_05T_ARTIFACT');ARTIFACT_06B_SOURCE=indexed('06b',EXPECTED_06B_INDEX_SHA256,'HAYFLOW_06B_ARTIFACT');ARTIFACT_06BB_SOURCE=indexed('06b-b',EXPECTED_06BB_INDEX_SHA256,'HAYFLOW_06BB_ARTIFACT');ARTIFACT_06BC_SOURCE=indexed('06b-c',EXPECTED_06BC_INDEX_SHA256,'HAYFLOW_06BC_ARTIFACT')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06bd_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.';print({'05t':str(ARTIFACT_05T_SOURCE),'06b':str(ARTIFACT_06B_SOURCE),'06b-b':str(ARTIFACT_06BB_SOURCE),'06b-c':str(ARTIFACT_06BC_SOURCE),'base':str(BASE_SOURCE)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 06b-d][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880;print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Preflight della matrice

Il preflight verifica autorità, split train-only, identità del bridge, stato congelato, bracci, checkpoint e contratti appaiati prima del training.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import NestedCouplingOptimizationScalingConfig,NestedCouplingOptimizationScalingForensic
cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_nested_coupling_optimization_scaling_forensic.yml').read_text());config=NestedCouplingOptimizationScalingConfig.from_mapping(cfg['nested_coupling_optimization_scaling_forensic'])
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_nested_coupling_optimization_scaling_forensic');assert not OUTPUT_DIR.exists(),f'Output gia presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=NestedCouplingOptimizationScalingForensic(bundle,OUTPUT_DIR,config,ARTIFACT_05T_SOURCE,ARTIFACT_06B_SOURCE,ARTIFACT_06BB_SOURCE,ARTIFACT_06BC_SOURCE,code_revision=REVISION);preflight=session.prepare_scaling_forensic()
display({'valid':preflight['valid'],'roles':preflight['role_transition_counts'],'arms':preflight['arms'],'checkpoints':preflight['scaling_checkpoints'],'bridge_parameters':preflight['bridge_parameter_count'],'paired_seeds':preflight['paired_seed_count'],'same_initialization':preflight['same_initialization_within_seed'],'same_batches':preflight['same_minibatch_stream_within_seed'],'state_frozen':preflight['state_updater_frozen'],'execution':preflight['physical_parallelism']});assert preflight['valid'] and preflight['state_and_outcome_splits_read']==['train'] and not preflight['validation_state_accessed'] and not preflight['test_state_accessed']

## 3. Training sincronizzato

Una sola traiettoria per braccio produce tutti i budget. Il tracker stampa soltanto avanzamento e loss mediana: non vengono riversati tensori, RNG o snapshot nel browser Kaggle.

In [ ]:
training=session.train_synchronized_scaling_matrix();compact_scales={seed:round(row['state_gradient_scale'],4) for seed,row in training['reports'].items()};display({'valid':training['valid'],'device':training['device'],'arms':training['arms'],'gradient_scales':compact_scales,'same_batches':training['same_minibatch_stream_within_seed'],'state_retraining':training['state_updater_retraining_performed']});assert training['valid'] and not training['state_updater_retraining_performed']

## 4. Confronti fissi e ricorrenza annidata

Tutti i budget erano preregistrati. Lo sviluppo train-derived non seleziona modelli; misura i contrasti. La ricorrenza aggiorna gli STATE per 8 ms mantenendo il voltaggio teacher come condizione al bordo.

In [ ]:
try:
 development=session.evaluate_fixed_budget_matrix();rollout=session.evaluate_final_nested_rollouts();final_report=session.finalize_scaling_forensic(training,development,rollout)
finally:
 session.close()
display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'gates':final_report['gate_checks'],'median_contrasts':{k:round(v,4) for k,v in final_report['median_contrasts'].items()},'questions':final_report['multiple_questions_answered_in_one_matrix'],'next_step':final_report['next_step']});assert final_report['valid'] and final_report['component_decision_grade'] and not final_report['full_training_authorized']

## 5. Crea e scarica lo ZIP

Downloader browser stabile del progetto: base64 → Blob → click locale.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_nested_coupling_optimization_scaling_forensic','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})